## 5.2 跳频、接入与功率控制原理

在上一节中，我们了解了本章的学习目标和前置要求。本节介绍星闪 SLE 标准中三项高级信道适应技术的核心原理。

本节学习大纲如下：

- 双节点架构：G/T 角色与 SleNode 组件
- 跳频扩频：跳频原理及瑞利分布
- 接入建链：六阶段流程与 G/T 角色协商
- 功率控制：三类信令与四种场景

---

### 1. 双节点系统架构

星闪 SLE 标准定义了 G 节点（Grant）和 T 节点（Terminal）两种角色。在 SleNode 仿真中，每个节点内部集成了 MAC 层和 PHY 层的完整功能栈：

**SleNode 内部组件：**

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
    <tr>
        <th style="border:1px solid #ccc; padding:8px; text-align:left; min-width:220px;">组件</th>
        <th style="border:1px solid #ccc; padding:8px; text-align:left; min-width:80px;">所属层</th>
        <th style="border:1px solid #ccc; padding:8px; text-align:left; min-width:280px;">功能</th>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">LinkManager</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">MAC</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">链路状态机（8 状态）</td>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">BroadcasterAccessManager</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">MAC</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">广播帧构建与接入管理</td>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">ScheduleManager</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">MAC</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">时隙/事件组调度</td>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">ARQ / FlowController</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">MAC</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">重传控制与流控背压</td>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">LinkQualityTracker</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">MAC</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">滑动窗口 FER 统计与 MCS 建议</td>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">PowerController</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">MAC</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">发射功率管理</td>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">SecurityManager</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">MAC</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">配对与加密上下文</td>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">FreqTable + PRNG</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">PHY</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">跳频序列生成</td>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">TxConfig + mac_to_iq</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">PHY</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">发射管线</td>
    </tr>
</table>


**双节点通信流程：**

```
G 节点                            T 节点
  |                                 |
  |-- start_advertising() --------> |  广播帧
  |                                 |-- start_scanning()
  |                                 |-- connect(g_addr)
  |-- accept_connection(t_addr) --> |  链路建立
  |                                 |
  |-- send(data) + transmit() ----> |  数据帧 (IQ)
  |                            AWGN |
  |                                 |-- receive(iq)
  |<-- process_feedback(success) -- |
  |                                 |
  |-- disconnect() ---------------> |  断开
```

---

### 2. 跳频（Frequency Hopping）

跳频是一种扩频通信技术，收发双方按照相同的伪随机序列在多个频点上快速切换。星闪 SLE 标准（TXS-10002-2025 6.10.3）定义了完整的跳频机制，核心优势在于**频率分集**——将深衰落的风险分散到多个独立信道，避免单个信道持续衰落导致链路中断。以 2.4 GHz 频段为例：中心频率 $F_c = 2402 + N$ MHz（$N = 0..78$），共 79 个射频信道。其中物理信道号 76/77/78 为广播信道，0-75 为数据信道。

跳频是最常用的扩频方式之一，其工作原理是指收发双方传输信号的载波频率按照预定规律进行离散变化的通信方式，也就是说，通信中使用的载波频率受伪随机变化码的控制而随机跳变。从通信技术的实现方式来说，"跳频"是一种用码序列进行多频频移键控的通信方式，也是一种码控载频跳变的通信系统。从时域上来看，跳频信号是一个多频率的频移键控信号；从频域上来看，跳频信号的频谱是一个在很宽频带上以不等间隔随机跳变的。下图是跳频在实际应用中的工作原理图：

<img src="./images/hop.png" width="700">

本仿真框架工作于复基带——信号始终是 $I+jQ$ 的复序列，不涉及射频载波 $e^{j2\pi f_c t}$。这意味着不存在"切换到不同载波频率"这一物理操作。因此仿真中用每帧更换 Rayleigh 信道种子来等效跳频：不同种子 → 独立信道系数 $h$ → 统计上与跳到不同频率（≥ 相干带宽时信道独立）等价，让实验聚焦于分集增益本身。

---

### 2. 瑞利（Rayleigh）分布 ###

瑞利分布是一个均值为0，方差为σ²的平稳窄带高斯过程，其包络的一维分布是瑞利分布。其表达式及概率密度如图所示。瑞利分布是最常见的用于描述平坦衰落信号接收包络或独立多径分量接受包络统计时变特性的一种分布类型。两个正交高斯噪声信号之和的包络服从瑞利分布。

<img src="./images/Rayleigh.png" width="300">

瑞利分布应用于通信工程，例如在独立和相关瑞利衰落信道上进行仿真。在多径瑞利衰落信道中，瑞利分布用于功率控制信号的二阶统计，这些统计可用于信道估计和信道编码方案的设计与评估。

---

### 3. 接入建链流程

星闪 SLE 的接入建链是一个六阶段握手过程（标准 7.1），涉及广播者（Broadcaster）和发起者（Initiator）两个角色的交互。

#### 3.1 六阶段流程

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
    <tr>
        <th style="border:1px solid #ccc; padding:8px; text-align:left; min-width:60px;">阶段</th>
        <th style="border:1px solid #ccc; padding:8px; text-align:left; min-width:150px;">方向</th>
        <th style="border:1px solid #ccc; padding:8px; text-align:left; min-width:180px;">动作</th>
        <th style="border:1px solid #ccc; padding:8px; text-align:left; min-width:320px;">关键内容</th>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">(a)</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">广播者 → 发起者</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">发送可扩展广播帧</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">AccessBasicInfo、SMF 参数、带宽/MCS 能力</td>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">(b)</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">发起者 → 广播者</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">发送接入请求</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">G/T 角色偏好、协商标志、支持的帧/带宽/MCS/导频</td>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">(c)</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">广播者 → 发起者</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">发送接入响应</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">接受/拒绝条目、分配的链路参数</td>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">(d)</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">发起者</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">接收响应</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">解析参数、配置调度器</td>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">(e)</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">双向</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">数据链路建立</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">链路状态切换为 CONNECTED</td>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">(f)</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">双向</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">安全管理（可选）</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">配对与加密</td>
    </tr>
</table>


#### 3.2 G/T 角色协商

每个节点可以表达 G 节点（Grant，调度主导方）或 T 节点（Terminal，终端）的偏好。当双方都不可协商时，默认发起者→G、广播者→T。协商结果决定后续的调度方向。


---

### 4. 发射功率控制

功率控制是闭环的发射功率调节机制（标准 7.2.13），通过三类信令实现对端与本端的功率协商与调整，在保证链路质量的前提下降低干扰和功耗。

#### 4.1 三类功率控制信令

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
    <tr>
        <th style="border:1px solid #ccc; padding:8px; text-align:left; min-width:200px;">信令</th>
        <th style="border:1px solid #ccc; padding:8px; text-align:left; min-width:130px;">Data Type Index</th>
        <th style="border:1px solid #ccc; padding:8px; text-align:left; min-width:80px;">长度</th>
        <th style="border:1px solid #ccc; padding:8px; text-align:left; min-width:420px;">关键字段</th>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">PowerControlRequest</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">0x0019</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">3 字节</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">tx_power_change（请求对方调整 dB 值）、sender_tx_power</td>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">PowerControlResponse</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">0x001A</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">4 字节</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">tx_power_change（实际调整量）、sender_min/max_power、acceptable_power_reduction</td>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">PowerChangeIndication</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">0x001B</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">4 字节</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">tx_power_change（单边变更值）、sender_tx_power、min/max 标志</td>
    </tr>
</table>


#### 4.2 四种应用场景

- **场景 a（对端请求）**：收到 PowerControlRequest → `handle_request()` 调整本端功率 → 回复 PowerControlResponse
- **场景 b（本端查询）**：发送 PowerControlRequest(tx_power_change=0) → 根据对端 Response 中的 acceptable_power_reduction 决策
- **场景 c（功率管理）**：启动 `power_management` → 主动发送 PowerChangeIndication 告知对端功率变化
- **场景 d（功率查询）**：发送 tx_power_change=0 的 Request → 查询对端当前功率状态

#### 4.3 本次仿真使用的简化自适应算法

可执行以下代码查看：

In [ ]:
!cat -n src/nearlink_sdr/sim/link_sim.py | sed -n "4285,4368p"

本仿真不直接使用上述三类标准信令，而是实现简化版的闭环功率控制。算法由以下步骤组成：

**① 连续失败检测**

每帧发送后检查 CRC 结果。成功 → `consec_fails = 0`；失败 → `consec_fails += 1`。引入连续失败计数（而非每次失败立即升功率）的目的是避免因单次噪声波动误触发功率上调，仅当信道条件持续恶化时才响应。

**② 阈值触发升功率**

```python
if consec_fails >= threshold:              # 例如 threshold=3，连续 3 次失败
    tx_power = min(tx_power + 2.0, 15.0)   # 升 2 dB，上限 15 dBm
    consec_fails = 0                        # 升完后重置计数
```

**③ IQ 幅值缩放——功率变化如何作用于信号**

这是功率自适应的核心物理环节。发射功率 $P_{tx}$（dBm）通过幅值增益 $g = 10^{P_{tx}/20}$ 乘到 IQ 信号上：

```python
gain = 10.0 ** (tx_power / 20.0)   # dBm → 线性幅值增益
scaled_iq = tx.iq * gain           # 功率作用于信号
```

这样 $P_{tx}$ 每升 2 dB，信号幅值放大 $10^{2/20} \approx 1.26$ 倍。

**④ 等效 SNR 计算**

本仿真中是根据 SNR 和信号功率来计算噪声功率的，所有为了让信号功率不会引起噪声功率的改变，引入了等效 SNR .发射功率提升要增大接收 SNR：

$$\text{SNR}_{\text{eff}} = \text{base\_snr\_db} + P_{tx} \quad (\text{dB})$$

```python
eff_snr_db = base_snr_db + tx_power
snr_linear = 10.0 ** (eff_snr_db / 10.0)
noise_std = np.sqrt(1.0 / (2.0 * snr_linear))
noise = noise_std * (standard_normal + 1j * standard_normal)
```

**⑤ 功率钳位**

`PowerController` 自动将功率限制在 [min_power_dbm, max_power_dbm] 范围内，防止功率无限上升或降到硬件极限以下。

**完整闭环流程：**

```
发送帧 → CRC 结果 → 失败? → consec_fails++
                  → 成功? → consec_fails = 0
       ↓
consec_fails >= 3? → 是 → tx_power += 2 dB (钳位在 15 dBm)
                  → 否 → 保持
       ↓
新 tx_power → gain = 10^(tx_power/20) → 缩放 IQ → 等效 SNR 提高 → 下一帧
```


---
### 练习

1. 跳频仿真中用"每帧独立 Rayleigh 种子"来等效跳频分集，这是因为：
   - A. 仿真框架工作于复基带，不涉及载波频率操作
   - B. Rayleigh 种子比 PRNG 计算更快
   - C. 独立随机种子在统计上与跳频到不同频率的信道独立性等价
   - D. A 和 C 都正确

2. 接入建链六阶段中，G/T 角色协商发生在哪个阶段？
   - A. 阶段 (a) 广播帧发送时
   - B. 阶段 (b) 接入请求中，发起者声明 G/T 偏好和协商标志
   - C. 阶段 (e) 数据链路建立后
   - D. 阶段 (c) 接入响应中，由广播者单方面决定

3. 关于 Rayleigh 衰落，以下说法正确的是：
   - A. Rayleigh 衰落是"某些频率噪声特别大"导致的
   - B. 深衰落是因为多径信号相位相反、互相抵消，与噪声无关
   - C. 固定信道下各帧的 Rayleigh 系数 h 每次随机变化
   - D. 跳频通过增大发射功率来对抗衰落

4. 功率自适应仿真中，为什么引入"连续失败阈值"而非每次失败立即升功率？
   - A. 为了减少 PowerControlRequest 信令的字节开销
   - B. 避免单次噪声波动误触发功率上调，仅当信道持续恶化才响应
   - C. 因为标准协议规定至少连续失败 3 次才能发送功率控制信令
   - D. 为了让功率轨迹图呈现更明显的阶梯形状


执行以下代码获取答案


In [ ]:
!cat answer/05.02_answer.txt
